In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/content/insurance.csv")
df.shape
# dataset yuklash va o'chamini ko'rish

(1338, 7)

In [3]:
df.head()
# dataset boshidan 5 ta qatorni ko'rish

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df.info()
# dataset ma'lumot turlarini ko'rish

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [5]:
df.isnull().sum()
# bo'sh qiymatlar borligini tekshiramiz, lekin bunda yo'q ekan
# ya'ni hamma ma'lumotlar to'liq ekan.

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [6]:
df.duplicated().sum()
# takrorlangan qiymatlar borligini tekshiramiz, bizga bitta bor ekan

np.int64(1)

In [7]:
df.describe()
# describe faqat sonli ustunlarni statistilasini hisoblaydi
# by default, df.describe() only calculates statistics for numerical columns.
# It completely ignores string, object, and categorical columns (like addresses, categories, or names)
# if your dataset contains a mix of both types.

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [11]:
# category bo'yicha taqsimoti

for col in ['sex', 'smoker', 'region']:
  print(f"\n{col} bo'yicha taqsimot:")
  print(df[col].value_counts())


sex bo'yicha taqsimot:
sex
male      676
female    662
Name: count, dtype: int64

smoker bo'yicha taqsimot:
smoker
no     1064
yes     274
Name: count, dtype: int64

region bo'yicha taqsimot:
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [9]:
print(df.groupby('sex')['charges'].agg(['count', 'mean', 'median']))
# jins bo'yicha o'rtacha xarajat hisoblandi.

        count          mean      median
sex                                    
female    662  12569.578844  9412.96250
male      676  13956.751178  9369.61575


In [12]:
print(df.groupby('smoker')["charges"].agg(['count', 'mean', 'median']))
# chekish bo'yicha o'rtacha xarajat hisoblanadi.

        count          mean       median
smoker                                  
no       1064   8434.268298   7345.40530
yes       274  32050.231832  34456.34845


In [13]:
print("\n--- Korrelyatsiya (sonli ustunlar) ---")
display(df.select_dtypes(include=[np.number]).corr())


--- Korrelyatsiya (sonli ustunlar) ---


,age,bmi,children,charges
age,1.000000,0.109272,0.042469,0.299008
bmi,0.109272,1.000000,0.012759,0.198341
children,0.042469,0.012759,1.000000,0.067998
charges,0.299008,0.198341,0.067998,1.000000


In [15]:
# 1. Duplicates (Takrorlarni tekshirish va o'chirish)
print("Tozalashdan oldingi shape:", df.shape)
duplicates_count = df.duplicated().sum()
print(f"Takroriy qatorlar soni: {duplicates_count}")

if duplicates_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Takrorlar o'chirildi.")

print("Tozalashdan keyingi shape:", df.shape)

# 2. Outliers tahlili (IQR usuli bilan bmi va charges)
for col in ['bmi', 'charges']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"\n{col} ustunidagi outlierlar soni: {len(outliers)} ({round(len(outliers)/len(df)*100, 2)}%)")
    print(f"Yuqori chegara: {round(upper_bound, 2)}, Maksimal qiymat: {round(df[col].max(), 2)}")

Tozalashdan oldingi shape: (1337, 7)
Takroriy qatorlar soni: 0
Tozalashdan keyingi shape: (1337, 7)

bmi ustunidagi outlierlar soni: 9 (0.67%)
Yuqori chegara: 47.32, Maksimal qiymat: 53.13

charges ustunidagi outlierlar soni: 139 (10.4%)
Yuqori chegara: 34524.78, Maksimal qiymat: 63770.43


In [16]:
# 1. Semizlik belgisi (BMI >= 30)
df['is_obese'] = (df['bmi'] >= 30).astype(int)

# 2. Bolasi yo'q / yolg'iz belgisi (Titanic'dagi is_alone kabi)
df['is_alone'] = (df['children'] == 0).astype(int)

# 3. Yosh toifalari (age_group)
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 35, 50, 100],
    labels=['young', 'middle_age', 'senior']
)

# 4. Semiz va chekuvchi (kombinatsiyalangan belgi)
df['obese_smoker'] = ((df['bmi'] >= 30) & (df['smoker'] == 'yes')).astype(int)

# Yangi feature'larning xarajatga ta'sirini tekshiramiz
print("--- Yangi ustunlar bo'yicha charges o'rtachasi ---")
print("\nSemizlik bo'yicha:")
print(df.groupby('is_obese')['charges'].mean().round(2))

print("\nSemiz + Chekuvchi bo'yicha:")
print(df.groupby('obese_smoker')['charges'].mean().round(2))

print("\nYosh toifalari bo'yicha:")
print(df.groupby('age_group', observed=False)['charges'].mean().round(2))

print("\nYangi qatorlar va ustunlar ko'rinishi:")
display(df[['age', 'bmi', 'is_obese', 'is_alone', 'age_group', 'obese_smoker', 'charges']].head())

--- Yangi ustunlar bo'yicha charges o'rtachasi ---

Semizlik bo'yicha:
is_obese
0    10713.67
1    15572.04
Name: charges, dtype: float64

Semiz + Chekuvchi bo'yicha:
obese_smoker
0     9839.16
1    41557.99
Name: charges, dtype: float64

Yosh toifalari bo'yicha:
age_group
young          9758.62
middle_age    14030.00
senior        18084.99
Name: charges, dtype: float64

Yangi qatorlar va ustunlar ko'rinishi:


,age,bmi,is_obese,is_alone,age_group,obese_smoker,charges
0,19,27.900,0,1,young,0,16884.92400
1,18,33.770,1,0,young,0,1725.55230
2,28,33.000,1,0,young,0,4449.46200
3,33,22.705,0,1,young,0,21984.47061
4,32,28.880,0,1,young,0,3866.85520


In [22]:
from sklearn.preprocessing import LabelEncoder
df_encoded = df.copy()

le = LabelEncoder()
# Binary ustunlar Label encoding ( Sex va Smoker )
df_encoded['sec_encoded'] = le.fit_transform(df_encoded['sex']) # female = 0, male = 1
df_encoded['somker_encoded'] = le.fit_transform(df_encoded['smoker']) # no = 0, yes = 1

# oridinal ustun: age_group(tartibli mapping)
age_map = {'young': 0, 'middle_age': 1, 'senior': 2}
df_encoded['age_group_encoded'] = df_encoded['age_group'].map(age_map)

# Nominal ustun: Region bo'yicha One-Hot Encoding(pf.get_dummies)
region_dummies = pd.get_dummies(df_encoded['region'], prefix='region', dtype=int)
df_encoded = pd.concat([df_encoded, region_dummies], axis=1)

# Asl matnli ustunlarni olib tashlaymiz
cols_to_drop = ['sex', 'smoker', 'region', 'age_group']
df_encoded = df_encoded.drop(columns = cols_to_drop)
print("Encodingdan keyingi untunlar ro'yxati:")
print(df_encoded.columns.tolist())
print('\nBirinchi 3 qator:')
display(df_encoded.head(3))

Encodingdan keyingi untunlar ro'yxati:
['age', 'bmi', 'children', 'charges', 'is_obese', 'is_alone', 'obese_smoker', 'sec_encoded', 'somker_encoded', 'age_group_encoded', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']

Birinchi 3 qator:


,age,bmi,children,charges,is_obese,is_alone,obese_smoker,sec_encoded,somker_encoded,age_group_encoded,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.90,0,16884.9240,0,1,0,0,1,0,0,0,0,1
1,18,33.77,1,1725.5523,1,0,0,1,0,0,0,0,1,0
2,28,33.00,3,4449.4620,1,0,0,1,0,0,0,0,1,0


In [23]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_scaled = df_encoded.copy()

# Agar imlo xatosi bo'lgan bo'lsa, to'g'rilab olamiz
df_scaled = df_scaled.rename(columns={
    'sec_encoded': 'sex_encoded',
    'somker_encoded': 'smoker_encoded'
})

# 1. MinMax Scaler: age va children uchun
minmax_cols = ['age', 'children']
minmax_scaler = MinMaxScaler()
df_scaled[[c + '_minmax' for c in minmax_cols]] = minmax_scaler.fit_transform(df_scaled[minmax_cols])

# 2. Standard Scaler: bmi uchun (outlierlarga chidamli)
standard_scaler = StandardScaler()
df_scaled['bmi_std'] = standard_scaler.fit_transform(df_scaled[['bmi']])

# Asl o'lchovdagi age, children, bmi ustunlarini saqlash yoki o'chirish
# Modelga faqat masshtablanganlarini kiritish uchun:
features_to_drop = ['age', 'children', 'bmi']
df_scaled_final = df_scaled.drop(columns=features_to_drop)

print("Scaling'dan keyingi ustunlar:")
print(df_scaled_final.columns.tolist())

print("\nMasshtablangan ustunlar statistikasi (describe):")
display(df_scaled_final[['age_minmax', 'children_minmax', 'bmi_std']].describe().round(3))

print("\nBirinchi 3 qator:")
display(df_scaled_final.head(3))

Scaling'dan keyingi ustunlar:
['charges', 'is_obese', 'is_alone', 'obese_smoker', 'sex_encoded', 'smoker_encoded', 'age_group_encoded', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest', 'age_minmax', 'children_minmax', 'bmi_std']

Masshtablangan ustunlar statistikasi (describe):


,age_minmax,children_minmax,bmi_std
count,1337.000,1337.000,1337.000
mean,0.461,0.219,-0.000
std,0.305,0.241,1.000
min,0.000,0.000,-2.411
25%,0.196,0.000,-0.717
50%,0.457,0.200,-0.043
75%,0.717,0.400,0.662
max,1.000,1.000,3.684



Birinchi 3 qator:


,charges,is_obese,is_alone,obese_smoker,sex_encoded,smoker_encoded,age_group_encoded,region_northeast,region_northwest,region_southeast,region_southwest,age_minmax,children_minmax,bmi_std
0,16884.9240,0,1,0,0,1,0,0,0,0,1,0.021739,0.0,-0.453160
1,1725.5523,1,0,0,1,0,0,0,0,1,0,0.000000,0.2,0.509422
2,4449.4620,1,0,0,1,0,0,0,0,1,0,0.217391,0.6,0.383155


In [24]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

def preprocess_insurance(df_raw):
    """
    Medical Cost Personal Datasets uchun to'liq preprocessing pipeline:
    1. Tozalash (Duplicates, NaN tekshiruvi)
    2. Feature Engineering (is_obese, is_alone, age_group, obese_smoker)
    3. Encoding (LabelEncoder, mapping, pd.get_dummies)
    4. Scaling (MinMaxScaler va StandardScaler)
    """
    df = df_raw.copy()

    # 1. Tozalash
    df = df.drop_duplicates().reset_index(drop=True)
    if df.isnull().sum().sum() > 0:
        # Sonli ustunlarni mediana, kategorik ustunlarni moda bilan to'ldirish
        num_cols = df.select_dtypes(include=[np.number]).columns
        cat_cols = df.select_dtypes(exclude=[np.number]).columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].median())
        for col in cat_cols:
            df[col] = df[col].fillna(df[col].mode()[0])

    # 2. Feature Engineering
    df['is_obese'] = (df['bmi'] >= 30).astype(int)
    df['is_alone'] = (df['children'] == 0).astype(int)
    df['obese_smoker'] = ((df['bmi'] >= 30) & (df['smoker'] == 'yes')).astype(int)

    age_bins = [17, 35, 50, 100]
    age_labels = ['young', 'middle_age', 'senior']
    df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

    # 3. Encoding
    le = LabelEncoder()
    df['sex_encoded'] = le.fit_transform(df['sex'])
    df['smoker_encoded'] = le.fit_transform(df['smoker'])

    age_map = {'young': 0, 'middle_age': 1, 'senior': 2}
    df['age_group_encoded'] = df['age_group'].map(age_map)

    region_dummies = pd.get_dummies(df['region'], prefix='region', dtype=int)
    df = pd.concat([df, region_dummies], axis=1)

    # Keraksiz matnli ustunlarni olib tashlash
    df = df.drop(columns=['sex', 'smoker', 'region', 'age_group'])

    # 4. Scaling
    minmax = MinMaxScaler()
    df[['age_minmax', 'children_minmax']] = minmax.fit_transform(df[['age', 'children']])

    std_scaler = StandardScaler()
    df['bmi_std'] = std_scaler.fit_transform(df[['bmi']])

    # Asl o'lchovdagi ustunlarni olib tashlash
    df = df.drop(columns=['age', 'children', 'bmi'])

    return df

# Sinovdan o'tkazamiz
raw_df = pd.read_csv("insurance.csv")
ready_df = preprocess_insurance(raw_df)

print("Model uchun to'liq tayyor dataset shakli:", ready_df.shape)
display(ready_df.head())

Model uchun to'liq tayyor dataset shakli: (1337, 14)


,charges,is_obese,is_alone,obese_smoker,sex_encoded,smoker_encoded,age_group_encoded,region_northeast,region_northwest,region_southeast,region_southwest,age_minmax,children_minmax,bmi_std
0,16884.92400,0,1,0,0,1,0,0,0,0,1,0.021739,0.0,-0.453160
1,1725.55230,1,0,0,1,0,0,0,0,1,0,0.000000,0.2,0.509422
2,4449.46200,1,0,0,1,0,0,0,0,1,0,0.217391,0.6,0.383155
3,21984.47061,0,1,0,1,0,0,0,1,0,0,0.326087,0.0,-1.305052
4,3866.85520,0,1,0,1,0,0,0,1,0,0,0.304348,0.0,-0.292456
